# Session 1: OpenAI API Engineering with Structured Outputs (Student Notebook)

Welcome to Session 1 of Day 2! In this notebook, you will move beyond basic chat completions into production-grade API engineering. You will master structured output generation using JSON mode, function calling (tool use), and Pydantic validation — skills essential for building reliable agentic systems that produce machine-readable outputs.

## Learning Objectives

By the end of this session, you will be able to:

1. **Use streaming** and track token usage for cost management
2. **Generate structured JSON** outputs using `response_format` parameter
3. **Define and use function calling** (tool use) with the OpenAI API
4. **Validate LLM outputs** with Pydantic models for type safety
5. **Build extraction pipelines** that turn unstructured text into structured data
6. **Implement robust API wrappers** with retries and error handling

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import os
from pydantic import BaseModel, Field
from typing import Optional, List

# Ensure your OpenAI API key is set as an environment variable:
#   export OPENAI_API_KEY="sk-..."
# Or uncomment the line below and paste your key (not recommended for production):
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("All imports successful!")
print(f"OpenAI SDK version: {openai.__version__}")

---
## Demos (Follow Along)

The following five demos are fully coded. Run each cell, observe the output, and make sure you understand what is happening before moving on.

### Demo 1: OpenAI API Deep Dive — Streaming, Token Usage, and Finish Reasons

In production agentic systems you need to: (a) track token usage for cost management, (b) stream responses for real-time UX, and (c) inspect finish reasons to know if the model completed its response or was cut off.

In [ ]:
# Demo 1 - OpenAI API Deep Dive

client = openai.OpenAI()

# Part A: Token usage tracking
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain quantum computing in 3 sentences."}],
    max_tokens=150
)

print("=== Token Usage ===")
print(f"Prompt tokens    : {response.usage.prompt_tokens}")
print(f"Completion tokens: {response.usage.completion_tokens}")
print(f"Total tokens     : {response.usage.total_tokens}")
print(f"Finish reason    : {response.choices[0].finish_reason}")
print(f"\nResponse:\n{response.choices[0].message.content}")

# Part B: Streaming responses
print("\n=== Streaming Response ===")
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Count from 1 to 5, one number per line."}],
    max_tokens=50,
    stream=True
)

collected_text = ""
for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        token = chunk.choices[0].delta.content
        collected_text += token
        print(token, end="", flush=True)

print(f"\n\nFull collected text: {collected_text}")

### Demo 2: Structured Output with JSON Mode

By setting `response_format={"type": "json_object"}`, the model is guaranteed to return valid JSON. This is critical for agentic systems where downstream code must parse the output reliably. You **must** include the word "JSON" in your prompt when using this mode.

In [ ]:
# Demo 2 - Structured Output with JSON Mode

client = openai.OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant that outputs JSON."},
        {"role": "user", "content": "Give me information about Python as a programming language. Include: name, year_created, creator, paradigms (list), and a short description."}
    ],
    response_format={"type": "json_object"},
    max_tokens=300
)

raw_json = response.choices[0].message.content
print("Raw JSON response:")
print(raw_json)

# Parse and pretty-print
parsed = json.loads(raw_json)
print("\nParsed and formatted:")
print(json.dumps(parsed, indent=2))

# Access individual fields
print(f"\nLanguage: {parsed.get('name', 'N/A')}")
print(f"Created : {parsed.get('year_created', 'N/A')}")
print(f"Creator : {parsed.get('creator', 'N/A')}")

### Demo 3: Function Calling — Defining and Using Tools

Function calling lets the model decide **when** and **how** to call external tools. The flow is:
1. You define tool schemas and send them with the request
2. The model returns a `tool_calls` response (instead of regular content)
3. You execute the function locally
4. You send the result back to the model for a final answer

In [ ]:
# Demo 3 - Function Calling / Tool Use

client = openai.OpenAI()

# Step 1: Define the tools (functions) the model can call
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g., 'San Francisco'"
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# Step 2: Simulated function implementation
def get_weather(city, unit="celsius"):
    """Simulated weather function."""
    weather_data = {
        "San Francisco": {"temp": 18, "condition": "Foggy"},
        "New York": {"temp": 25, "condition": "Sunny"},
        "London": {"temp": 15, "condition": "Rainy"},
    }
    data = weather_data.get(city, {"temp": 20, "condition": "Unknown"})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9/5 + 32)
    return json.dumps({"city": city, "temperature": data["temp"], "unit": unit, "condition": data["condition"]})

# Step 3: Make the initial API call
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "What's the weather like in San Francisco?"}],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")

# Step 4: Check if the model wants to call a function
if message.tool_calls:
    tool_call = message.tool_calls[0]
    print(f"Function called: {tool_call.function.name}")
    print(f"Arguments: {tool_call.function.arguments}")

    # Step 5: Execute the function
    args = json.loads(tool_call.function.arguments)
    result = get_weather(**args)
    print(f"Function result: {result}")

    # Step 6: Send the result back to the model
    follow_up = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": "What's the weather like in San Francisco?"},
            message,
            {"role": "tool", "tool_call_id": tool_call.id, "content": result}
        ],
        tools=tools
    )
    print(f"\nFinal response:\n{follow_up.choices[0].message.content}")
else:
    print(f"Direct response: {message.content}")

### Demo 4: Pydantic-Based Response Validation

Pydantic models let you define the **exact shape** of the data you expect from the LLM, with type checking and constraints. If the LLM returns data that does not match, Pydantic will raise an error — catching problems before they propagate through your agent.

In [ ]:
# Demo 4 - Pydantic-Based Response Validation

client = openai.OpenAI()

# Step 1: Define Pydantic models for structured output
class MovieReview(BaseModel):
    title: str = Field(description="The movie title")
    year: int = Field(description="Release year")
    rating: float = Field(ge=0, le=10, description="Rating out of 10")
    genre: str = Field(description="Primary genre")
    summary: str = Field(description="One-sentence summary")
    recommended: bool = Field(description="Whether you recommend this movie")

# Step 2: Request structured output using JSON mode
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": (
            "You are a movie critic. Output your review as JSON with these exact fields: "
            "title (string), year (integer), rating (float 0-10), genre (string), "
            "summary (string), recommended (boolean)."
        )},
        {"role": "user", "content": "Review the movie 'Inception' by Christopher Nolan."}
    ],
    response_format={"type": "json_object"},
    max_tokens=200
)

raw = response.choices[0].message.content
print("Raw JSON:")
print(raw)

# Step 3: Validate with Pydantic
try:
    review = MovieReview.model_validate_json(raw)
    print("\nValidated MovieReview:")
    print(f"  Title      : {review.title}")
    print(f"  Year       : {review.year}")
    print(f"  Rating     : {review.rating}/10")
    print(f"  Genre      : {review.genre}")
    print(f"  Summary    : {review.summary}")
    print(f"  Recommended: {'Yes' if review.recommended else 'No'}")
except Exception as e:
    print(f"\nValidation error: {e}")

### Demo 5: Building a Structured Data Extraction Pipeline

This demo combines JSON mode and Pydantic into a reusable pipeline that extracts structured contact information from unstructured text. This pattern is the foundation for many agentic data-processing workflows.

In [ ]:
# Demo 5 - Building a Structured Data Extraction Pipeline

client = openai.OpenAI()

class ContactInfo(BaseModel):
    name: str = Field(description="Full name of the person")
    email: Optional[str] = Field(default=None, description="Email address if found")
    phone: Optional[str] = Field(default=None, description="Phone number if found")
    company: Optional[str] = Field(default=None, description="Company name if found")
    role: Optional[str] = Field(default=None, description="Job title or role if found")

def extract_contacts(text: str) -> List[ContactInfo]:
    """Extract structured contact information from unstructured text."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": (
                "Extract all contact information from the text. "
                "Return a JSON object with a 'contacts' key containing a list. "
                "Each contact should have: name, email (null if not found), "
                "phone (null if not found), company (null if not found), "
                "role (null if not found)."
            )},
            {"role": "user", "content": text}
        ],
        response_format={"type": "json_object"},
        max_tokens=500
    )

    data = json.loads(response.choices[0].message.content)
    contacts = []
    for item in data.get("contacts", []):
        try:
            contacts.append(ContactInfo(**item))
        except Exception as e:
            print(f"Skipping invalid contact: {e}")
    return contacts

# Test the extraction pipeline
sample_text = """
Hi, I'm Sarah Chen from TechCorp. You can reach me at sarah.chen@techcorp.com
or call (555) 123-4567. I'm the VP of Engineering.

Also, please loop in our CTO, James Rodriguez. His email is james.r@techcorp.com.

For billing questions, contact accounts@techcorp.com -- ask for Maria Lopez.
"""

print("Input text:")
print(sample_text)
print("=" * 60)
print("Extracted contacts:")
print("=" * 60)

contacts = extract_contacts(sample_text)
for i, contact in enumerate(contacts, 1):
    print(f"\nContact {i}:")
    print(f"  Name   : {contact.name}")
    print(f"  Email  : {contact.email or 'N/A'}")
    print(f"  Phone  : {contact.phone or 'N/A'}")
    print(f"  Company: {contact.company or 'N/A'}")
    print(f"  Role   : {contact.role or 'N/A'}")

---
## Tasks (Your Turn!)

Now it is your turn to practice. Each task below has a description, hints, and a code skeleton with `TODO` placeholders. Fill in the code to complete each task.

### Task 1: Build a Structured Entity Extractor Using JSON Mode

Create a function that takes a news article (string) and extracts structured entities — people, organizations, locations, and dates — returning the result as a parsed dictionary.

In [ ]:
# Task 1 - Build a Structured Entity Extractor Using JSON Mode

# TODO: Create a function that takes a news article (string) and extracts
# structured entities: people, organizations, locations, and dates.
# Return the result as a parsed dictionary.
#
# Hint: Use response_format={"type": "json_object"}
# Hint: Define the expected JSON schema in the system message
# Hint: Use json.loads() to parse the response

def extract_entities(article_text):
    """Extract named entities from a news article using JSON mode."""
    # YOUR CODE HERE
    pass


# Test your function
# sample_article = (
#     "Apple CEO Tim Cook announced at WWDC in San Jose on June 10, 2024 that "
#     "the company would partner with OpenAI to integrate ChatGPT into iOS. "
#     "Microsoft CEO Satya Nadella responded from Redmond, praising the collaboration."
# )
# entities = extract_entities(sample_article)
# print(json.dumps(entities, indent=2))

### Task 2: Implement a Calculator Tool with Function Calling

Create a calculator that uses OpenAI function calling. Define a `calculate` tool with parameters for the operation and two numbers, implement the actual function, and handle the full function-calling loop.

In [ ]:
# Task 2 - Implement a Calculator Tool with Function Calling

# TODO: Create a calculator that uses OpenAI function calling.
# 1. Define a "calculate" tool with parameters: operation (add/subtract/multiply/divide)
#    and two numbers (a, b)
# 2. Implement the actual calculate function
# 3. Handle the full function calling loop (request -> tool call -> execute -> respond)
#
# Hint: Define the tool schema with operation as an enum
# Hint: Parse the function arguments with json.loads()
# Hint: Send the tool result back as a follow-up message with role="tool"

def calculate(operation, a, b):
    """Perform a math operation."""
    # YOUR CODE HERE
    pass

def ask_math_question(question):
    """Send a math question and handle function calling."""
    # YOUR CODE HERE
    pass


# Test your function
# print(ask_math_question("What is 42 multiplied by 17?"))
# print(ask_math_question("Divide 144 by 12"))

### Task 3: Create a Multi-Tool Agent with Automatic Tool Dispatch

Build an agent that has access to multiple tools (weather, calculator, time) and automatically dispatches to the right one based on the user's question.

In [ ]:
# Task 3 - Create a Multi-Tool Agent with Automatic Tool Dispatch

# TODO: Build an agent that has access to multiple tools and automatically
# dispatches to the right one based on the user's question.
# Tools to implement:
#   1. get_weather(city) - returns simulated weather data
#   2. calculate(operation, a, b) - performs math
#   3. get_time(timezone) - returns simulated current time
#
# Hint: Define all three tools in the tools list
# Hint: Create a dispatch dictionary mapping function names to implementations
# Hint: Handle the case where the model calls a tool AND the case where it responds directly

tools = [
    # YOUR TOOL DEFINITIONS HERE
]

def dispatch_tool_call(tool_name, arguments):
    """Route a tool call to the correct function."""
    # YOUR CODE HERE
    pass

def multi_tool_agent(user_message):
    """Process a user message with multi-tool support."""
    # YOUR CODE HERE
    pass


# Test your agent
# print(multi_tool_agent("What's the weather in London?"))
# print(multi_tool_agent("Calculate 25 times 4"))
# print(multi_tool_agent("What time is it in Tokyo?"))

### Task 4: Build a Robust API Client with Retries, Validation, and Streaming

Build a production-grade API client class that wraps OpenAI chat completions with automatic retry logic, Pydantic validation, streaming support, and token usage tracking.

In [ ]:
# Task 4 - Build a Robust API Client with Retries, Validation, and Streaming

# TODO: Build a production-grade API client class that:
# 1. Wraps OpenAI chat completions with automatic retry logic (max 3 retries)
# 2. Validates responses against a Pydantic model when provided
# 3. Supports both streaming and non-streaming modes
# 4. Logs token usage for cost tracking
#
# Hint: Use a try/except loop with exponential backoff (time.sleep(2 ** attempt))
# Hint: Use Pydantic's model_validate_json() for validation
# Hint: Track cumulative token usage in instance variables

import time

class RobustLLMClient:
    def __init__(self, model="gpt-4o-mini", max_retries=3):
        """Initialize the client with retry settings."""
        # YOUR CODE HERE
        pass

    def call(self, messages, response_model=None, stream=False, **kwargs):
        """Make a robust API call with retries and optional validation."""
        # YOUR CODE HERE
        pass

    def get_usage_stats(self):
        """Return cumulative token usage statistics."""
        # YOUR CODE HERE
        pass


# Test your client
# client = RobustLLMClient()
# result = client.call(
#     messages=[{"role": "user", "content": "Describe Python in JSON: name, year, creator"}],
#     stream=False
# )
# print(result)
# print(client.get_usage_stats())

---
## Summary

In this session you learned production-grade OpenAI API engineering skills:

1. **API Deep Dive** -- Streaming responses for real-time output, tracking token usage for cost management, and understanding finish reasons.
2. **JSON Mode** -- Using `response_format={"type": "json_object"}` to get reliable, parseable structured data from the model.
3. **Function Calling** -- Defining tools that the model can invoke, handling the request-execute-respond loop that powers agentic tool use.
4. **Pydantic Validation** -- Using Pydantic models to validate and parse LLM outputs with type safety and automatic error detection.
5. **Extraction Pipelines** -- Combining JSON mode and Pydantic into reusable pipelines that extract structured data from unstructured text.

**Up Next:** In Session 2, we will move from raw API calls to LangChain, learning how to compose modular chains, integrate tools, and build retrieval-augmented generation (RAG) systems.